# BeautifulSoup Command Reference

This notebook summarizes the BeautifulSoup commands used in the Day 2 scraping notebooks.

It is not a complete BeautifulSoup manual. It is a practical reference for the commands you need to understand the course scripts:

- parse HTML into a `soup` object;
- inspect HTML structure;
- select elements;
- extract visible text;
- extract attributes;
- handle missing elements;
- use a few more advanced search patterns.


## 1. Imports and Example HTML

We use a small HTML fragment that looks like a simplified news/search result page.

The same commands work with real HTML from `requests.get(...).text`; the only difference is that real pages are usually much larger and messier.


In [ ]:
from bs4 import BeautifulSoup

html = """
<html>
  <head>
    <title>Example News Search</title>
  </head>
  <body>
    <main id="results" data-page="search-results">
      <article class="result card featured" data-id="news-001" data-kind="result-article">
        <p class="section-label">Politics</p>
        <h2 class="headline">
          <a href="/news/dsa-election-platforms">Platforms face election transparency questions</a>
        </h2>
        <p class="summary">Researchers are watching how platforms report moderation decisions.</p>
        <time class="published" datetime="2026-06-25">25 June 2026</time>
        <img src="/images/platforms.jpg" alt="Laptop showing social media feeds">
        <ul class="tags">
          <li><a class="tag" href="/topics/platforms">platforms</a></li>
          <li><a class="tag" href="/topics/elections">elections</a></li>
        </ul>
      </article>

      <article class="result card" data-id="news-002" data-kind="result-article">
        <p class="section-label">Technology</p>
        <h2 class="headline">
          <a href="/news/api-access">Researchers discuss API access</a>
        </h2>
        <p class="summary">Access routes shape which platform data can be studied.</p>
        <time class="published" datetime="2026-06-26">26 June 2026</time>
        <ul class="tags">
          <li><a class="tag" href="/topics/apis">APIs</a></li>
          <li><a class="tag" href="/topics/research-methods">research methods</a></li>
        </ul>
      </article>
    </main>
  </body>
</html>
"""


## 2. `BeautifulSoup(html, "html.parser")`

`BeautifulSoup(...)` parses HTML text into a searchable object.

Before this step, `html` is only a string. After this step, Python can search for tags, classes, ids, links, tables, and attributes.


In [ ]:
soup = BeautifulSoup(html, "html.parser")

type(soup)


## 3. `.prettify()`

`.prettify()` prints the HTML with indentation. This is useful for inspection, especially when the original HTML is one long line.

Use it for diagnosis, not for data extraction.


In [ ]:
print(soup.prettify()[:1000])


## 4. `.select()`

`.select("CSS selector")` returns **all** matching elements as a list.

Use `.select()` when you expect several things: many paragraphs, many links, many result cards, many table rows.


In [ ]:
# Select all article cards.
cards = soup.select("article.result")

print(type(cards))
print("Number of cards:", len(cards))

for card in cards:
    print(card.get("data-id"))


## 5. `.select_one()`

`.select_one("CSS selector")` returns the **first** matching element, or `None` if nothing matches.

Use `.select_one()` when you expect one value: one title, one author, one date, one image per record.


In [ ]:
first_card = soup.select_one("article.result")
headline = first_card.select_one(".headline a")

print(headline)


## 6. `.get_text()`

`.get_text()` extracts visible text from an HTML element.

In the course scripts we usually use:

```python
element.get_text(" ", strip=True)
```

This means: extract visible text, join internal pieces with spaces, and remove leading/trailing whitespace.


In [ ]:
headline_text = headline.get_text(" ", strip=True)
summary_text = first_card.select_one(".summary").get_text(" ", strip=True)

print(headline_text)
print(summary_text)


## 7. `.get("attribute")`

`.get("attribute")` extracts an attribute value from an HTML element.

Common attributes in scraping:

- `href`: link target;
- `src`: image or media file location;
- `alt`: image description;
- `datetime`: machine-readable date;
- `class`: class labels;
- `id`: unique page identifier;
- `data-*`: custom metadata attributes.


In [ ]:
link = first_card.select_one(".headline a")
image = first_card.select_one("img")
date = first_card.select_one("time.published")

print("href:", link.get("href"))
print("src:", image.get("src"))
print("alt:", image.get("alt"))
print("datetime:", date.get("datetime"))
print("classes:", first_card.get("class"))
print("data-id:", first_card.get("data-id"))


## 8. `.get("attribute", default)`

You can provide a default value if the attribute is missing.

This is useful for class labels because BeautifulSoup returns classes as a list. If an element has no class, `element.get("class", [])` gives an empty list instead of `None`.


In [ ]:
print("Classes on first card:", first_card.get("class", []))
print("Missing attribute with default:", first_card.get("data-missing", "not available"))

if "featured" in first_card.get("class", []):
    print("This card is featured.")


## 9. Defensive Extraction

Real pages are inconsistent. An element may be missing.

This pattern prevents the script from crashing:

```python
value = element.get_text(" ", strip=True) if element else None
```

It means: if the element exists, extract text; otherwise store `None`.


In [ ]:
missing_author = first_card.select_one(".author")

author = missing_author.get_text(" ", strip=True) if missing_author else None

print(author)


## 10. `.title`

`soup.title` is a shortcut for the page's `<title>` element.

It is useful for quick inspection, but for careful scraping you usually select the exact page title or record title you need.


In [ ]:
print(soup.title)
print(soup.title.get_text(" ", strip=True))


## 11. `.name`

`.name` gives the tag name of an element, such as `article`, `a`, `p`, `time`, or `img`.

This is mostly useful for inspection and debugging.


In [ ]:
element = soup.select_one("time.published")

print(element)
print(element.name)


## 12. `.attrs`

`.attrs` gives all attributes on an element as a dictionary.

This is useful when you do not yet know which attributes a page uses.


In [ ]:
print(first_card.attrs)
print(image.attrs)


## 13. `.find()` and `.find_all()`

Most course examples use `.select()` and `.select_one()` because CSS selectors match what you see in browser developer tools.

BeautifulSoup also has `.find()` and `.find_all()`.

- `.find(...)` returns the first match.
- `.find_all(...)` returns all matches.

They are useful when CSS selectors are not flexible enough, especially if you want to search with a Python function.


In [ ]:
# Find the first <article> tag.
print(soup.find("article"))

# Find all <article> tags.
print(len(soup.find_all("article")))


## 14. `.find_all()` with a Function

Sometimes we want a condition that is easier to express in Python than in a CSS selector.

Example: find elements where **any attribute value** contains the word `result`.


In [ ]:
def any_attribute_value_contains(element, needle):
    # element.attrs is a dictionary of all attributes on the element.
    for value in element.attrs.values():
        # Class attributes are often stored as lists, so turn lists into text.
        if isinstance(value, list):
            value_as_text = " ".join(value)
        else:
            value_as_text = str(value)

        if needle in value_as_text:
            return True

    return False

matching_elements = soup.find_all(
    lambda element: any_attribute_value_contains(element, "result")
)

for element in matching_elements:
    print(element.name, element.attrs)


## 16. Quick Reference Table

| Command | Use it for | Returns |
|---|---|---|
| `BeautifulSoup(html, "html.parser")` | parse HTML text | soup object |
| `.prettify()` | inspect nested HTML | formatted string |
| `.select("...")` | find all elements matching a CSS selector | list |
| `.select_one("...")` | find the first element matching a CSS selector | element or `None` |
| `.get_text(" ", strip=True)` | extract readable visible text | string |
| `.get("href")` | extract an attribute value | string/list or `None` |
| `.get("class", [])` | extract an attribute with a fallback | value or default |
| `.title` | inspect the `<title>` element | element or `None` |
| `.name` | inspect an element's tag name | string |
| `.attrs` | inspect all attributes | dictionary |
| `.find(...)` | find first match using BeautifulSoup syntax/function | element or `None` |
| `.find_all(...)` | find all matches using BeautifulSoup syntax/function | list |
